imports

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from ClassificationModel.src.models.classification_model import CancerClassificationModel
from ClassificationModel.src.utils.dataset_utils import load_dataset
from constants.classification.datasets_constants import DatasetConstants
from constants.classification.model_constants import ModelConstants
from utils.callbacks.notification_callback import NotificationCallback
from utils.class_weight_utils import calculate_class_weights
from tensorflow import keras
from ClassificationModel.src.data_processing.image_augmentation import ImageAugmentationPipeline

Loading the dataset and creating the model

In [3]:
dataset = load_dataset()

CHECKPOINT = ModelConstants.CHECKPOINT_FILE_PATH

model = CancerClassificationModel(dataset,
                                  (*DatasetConstants.IMAGE_SIZE, DatasetConstants.CHANNELS)
                                  ,checkpoint_path=CHECKPOINT)

augmenter = ImageAugmentationPipeline()

EPOCHS=50

Found 1013 files belonging to 4 classes.
Found 144 files belonging to 4 classes.
Found 610 files belonging to 4 classes.
Loading checkpoint from: Checkpoints/best_model.keras
✓ Model loaded successfully
  Input shape: (None, 224, 224, 1)
  Output classes: 4


defining callbacks

In [4]:
callbacks = [
    NotificationCallback(
        notifier=model.notifier,
        metrics_to_track=[
            ModelConstants.TRACKED_ACCURACY_METRIC,
            ModelConstants.TRACKED_LOSS_METRIC,
            ModelConstants.TRACKED_VAL_ACCURACY_METRIC,
            ModelConstants.TRACKED_VAL_LOSS_METRIC
        ]),
    keras.callbacks.ModelCheckpoint(
        filepath=ModelConstants.CHECKPOINT_FILE_PATH,
        save_best_only=True,
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC
    ),
    keras.callbacks.EarlyStopping(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        patience=15, 
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        factor=0.5,
        patience=5,  
        min_lr=1e-7
    )
]

In [5]:
# Calculate class weights to handle imbalance
class_weights = calculate_class_weights(
    dataset[DatasetConstants.TRAIN_SPLIT_NAME],
    class_names=dataset[DatasetConstants.CLASS_NAMES_KEY]
)

Class weights for handling imbalance:
  adenocarcinoma: 0.858
  large.cell.carcinoma: 1.178
  normal: 1.021
  squamous.cell.carcinoma: 0.993


2026-01-09 17:39:04.243754: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


training the model

In [6]:
history = model.train_model(
    epochs=EPOCHS, 
    callbacks=callbacks, 
    augment_train=True, 
    augmenter=augmenter,
    class_weight=class_weights
)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.9970 - auc: 1.0000 - loss: 0.0047 - precision: 0.9970 - recall: 0.9970 - val_accuracy: 0.9306 - val_auc: 0.9910 - val_loss: 0.2150 - val_precision: 0.9296 - val_recall: 0.9167 - learning_rate: 5.0000e-04
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.9970 - auc: 1.0000 - loss: 0.0058 - precision: 0.9970 - recall: 0.9970 - val_accuracy: 0.9167 - val_auc: 0.9896 - val_loss: 0.2369 - val_precision: 0.9167 - val_recall: 0.9167 - learning_rate: 5.0000e-04
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.9980 - auc: 1.0000 - loss: 0.0048 - precision: 0.9980 - recall: 0.9980 - val_accuracy: 0.9167 - val_auc: 0.9874 - val_loss: 0.2589 - val_precision: 0.9167 - val_recall: 0.9167 - learning_rate: 5.0000e-04
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.9970 - auc: 1.0000 - loss: 0.0043 - precision: 0.9970 - recall: 0.9970 - val_accuracy: 0.9167 - val_auc: 0.9875 - val_loss: 0.2526 -

evaluating the model

In [7]:

results = model.evaluate_model(present_metrics=True,send_message=True,save_confusion_matrix=True)

20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 349ms/step - accuracy: 0.5475 - auc: 0.7479 - loss: 2.0992 - precision: 0.5790 - recall: 0.5344
accuracy: 0.548
auc: 0.748
loss: 2.099
precision: 0.579
recall: 0.534


2026-01-09 17:51:55.493746: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



✓ Confusion matrix saved to: ClassificationModel/testing/results/confusion_matrix_20260109_175155.png

Classification Report:
                         precision    recall  f1-score   support

         adenocarcinoma      0.500     0.082     0.141       220
   large.cell.carcinoma      0.400     0.804     0.534       102
                 normal      0.847     0.926     0.885       108
squamous.cell.carcinoma      0.534     0.744     0.622       180

               accuracy                          0.548       610
              macro avg      0.570     0.639     0.545       610
           weighted avg      0.555     0.548     0.480       610

